In [4]:
import pandas as pd


import numpy as np

In [5]:
# 1. Leer el CSV
votaciones = pd.read_csv('C:/Users/TALIGENT/Documents/DS_CP/ds4p/LegisTrack/hcdn_votaciones_historico.csv')

# 2. Eliminar las columnas inútiles
votaciones = votaciones.drop(columns=['Unnamed: 0', '¿QUÉ DIJO?'])

# 3. Renombrar las columnas para no tener mayúsculas, espacios ni signos de interrogación
votaciones = votaciones.rename(columns={
    'DIPUTADO': 'diputado',
    'BLOQUE': 'bloque',
    'PROVINCIA': 'provincia',
    '¿CÓMO VOTÓ?': 'voto'
})

# 4. Convertir la fecha (que ahora es texto) a un formato de Fecha real (Datetime)
# Esto es vital para después poder filtrar por "Año" o "Mes" en el dashboard
votaciones['fecha_votacion'] = pd.to_datetime(votaciones['fecha_votacion'], format='%d/%m/%Y', errors='coerce')

# Vemos el resultado limpio
print(votaciones.head())
print("\nEstructura final del dataset:")
print(votaciones.info())

votaciones.head(8)

                            diputado                        bloque  \
0  ABDALA DE MATARAZZO, Norma Amanda    Frente Cívico por Santiago   
1                 ABRAHAM, Alejandro  Frente para la Victoria - PJ   
2                  AGUAD, Oscar Raúl          Unión Cívica Radical   
3               AGUILAR, Lino Walter            Compromiso Federal   
4             ALEGRE, Gilberto Oscar              Frente Renovador   

             provincia        voto  id_votacion  \
0  Santiago del Estero  AFIRMATIVO            1   
1              Mendoza  AFIRMATIVO            1   
2              Córdoba  AFIRMATIVO            1   
3             San Luis     AUSENTE            1   
4         Buenos Aires  AFIRMATIVO            1   

                                     titulo_proyecto fecha_votacion  
0  Régimen previsional especial de carácter excep...     2015-10-07  
1  Régimen previsional especial de carácter excep...     2015-10-07  
2  Régimen previsional especial de carácter excep...     2015-

,diputado,bloque,provincia,voto,id_votacion,titulo_proyecto,fecha_votacion
0,"ABDALA DE MATARAZZO, Norma Amanda",Frente Cívico por Santiago,Santiago del Estero,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07
1,"ABRAHAM, Alejandro",Frente para la Victoria - PJ,Mendoza,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07
2,"AGUAD, Oscar Raúl",Unión Cívica Radical,Córdoba,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07
3,"AGUILAR, Lino Walter",Compromiso Federal,San Luis,AUSENTE,1,Régimen previsional especial de carácter excep...,2015-10-07
4,"ALEGRE, Gilberto Oscar",Frente Renovador,Buenos Aires,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07
5,"ALFONSÍN, Ricardo",Unión Cívica Radical,Buenos Aires,AUSENTE,1,Régimen previsional especial de carácter excep...,2015-10-07
6,"ALONSO, Laura",Unión PRO,C.A.B.A.,AUSENTE,1,Régimen previsional especial de carácter excep...,2015-10-07
7,"ALONSO, María Luz",Frente para la Victoria - PJ,La Pampa,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07


In [13]:
import re
import pandas as pd

# ============================================================
# PASO 1 — Eliminar ruido procedimental
# ============================================================
patrones_ruido = [
    r'APARTAMIENTO DE REGLAMENTO',
    r'Apartamiento de [Rr]eglamento',
    r'Apartamiento del [Rr]eglamento',
    r'A N U L A D A',
    r'Plan de Labor Parlamentaria',
    r'Conjunto de proyectos',
    r'Conjunto de varios',
    r'^\s*-\s*Votación',
    r'MOCIÓN SOLICITADA POR',
    r'Moción solicitada por',
]
regex_ruido = '|'.join(patrones_ruido)
df = votaciones[~votaciones['titulo_proyecto'].str.contains(
    regex_ruido, regex=True, na=False
)].copy()
print(f"[1] Registros originales:          {len(votaciones):,}")
print(f"[1] Registros tras eliminar ruido: {len(df):,}")
print(f"[1] Ruido eliminado:               {len(votaciones)-len(df):,}")

# ============================================================
# PASO 2 — Extraer título base + fecha de sesión
# ============================================================
def extraer_titulo_base(titulo):
    if pd.isna(titulo):
        return titulo
    titulo = re.sub(
        r'\s*-\s*(Artículos?\s*\d+[°º]?[^–]*|En General[^–]*|En Particular[^–]*|Votación en General[^–]*)',
        '', titulo, flags=re.IGNORECASE
    )
    titulo = re.sub(r'\s*\d{2}/\d{2}/\d{4}\s*-\s*\d{2}:\d{2}\s*$', '', titulo)
    return titulo.strip()

df['titulo_base'] = df['titulo_proyecto'].apply(extraer_titulo_base)
df['fecha_votacion'] = pd.to_datetime(df['fecha_votacion'])
df['fecha_base'] = df['fecha_votacion'].dt.date

# fecha_sesion = primer día del grupo (diputado, titulo_base)
# resuelve cruces de medianoche
df['fecha_sesion'] = df.groupby(
    ['diputado', 'titulo_base']
)['fecha_base'].transform('min')

print(f"\n[2] Títulos únicos originales: {df['titulo_proyecto'].nunique():,}")
print(f"[2] Títulos base únicos:        {df['titulo_base'].nunique():,}")

# ============================================================
# PASO 3 — Flag de votación "En General"
# ============================================================
df['es_voto_general'] = df['titulo_proyecto'].str.contains(
    r'En General|EN GENERAL', regex=True, na=False
)
print(f"\n[3] Registros con voto 'En General': {df['es_voto_general'].sum():,}")

# ============================================================
# PASO 4 — Consolidación vectorizada
# KEY incluye fecha_sesion para aislar sesiones distintas
# con mismo titulo_base (ej: "Votación en General y Particular...")
# ============================================================
def moda_voto(s):
    m = s.mode()
    return m.iloc[0] if len(m) > 0 else 'ABSTENCIÓN'

cols_contexto = ['bloque', 'provincia', 'fecha_votacion']
KEY = ['diputado', 'titulo_base', 'fecha_sesion']

# 4a — Grupos con voto "En General"
consolidado_general = (
    df[df['es_voto_general']]
    .groupby(KEY)
    .agg(
        voto=('voto', moda_voto),
        **{col: (col, 'first') for col in cols_contexto}
    )
    .reset_index()
    .assign(fuente_consolidacion='en_general')
)

# 4b — Grupos SIN voto "En General"
pares_con_general = set(
    zip(consolidado_general['diputado'],
        consolidado_general['titulo_base'],
        consolidado_general['fecha_sesion'])
)
df_art = df[~df['es_voto_general']].copy()
df_art['_key'] = list(zip(
    df_art['diputado'],
    df_art['titulo_base'],
    df_art['fecha_sesion']
))
df_solo_art = df_art[~df_art['_key'].isin(pares_con_general)].drop(columns='_key')

consolidado_articulos = (
    df_solo_art
    .groupby(KEY)
    .agg(
        voto=('voto', moda_voto),
        **{col: (col, 'first') for col in cols_contexto}
    )
    .reset_index()
    .assign(fuente_consolidacion='moda_articulos')
)

# 4c — Unir y limpiar
df_consolidado = (
    pd.concat([consolidado_general, consolidado_articulos], ignore_index=True)
    .drop(columns=['fecha_sesion'])
)

# ============================================================
# PASO 5 — Eliminar categorías de voto no informativas
# ============================================================
votos_ruido = ['PRESIDENTE', 'PENDIENTE DE INCORPORACIÓN', 'SIN VOTAR']
df_consolidado = df_consolidado[~df_consolidado['voto'].isin(votos_ruido)].copy()

# ============================================================
# VERIFICACIÓN FINAL
# ============================================================
print(f"\n[4] Registros originales:          {len(votaciones):,}")
print(f"[4] Tras eliminar ruido:            {len(df):,}")
print(f"[4] Tras consolidar:                {len(df_consolidado):,}")
print(f"[4] Reducción total:                {len(votaciones)-len(df_consolidado):,}")
print(f"\n[4] Fuente de consolidación:")
print(df_consolidado['fuente_consolidacion'].value_counts())
print(f"\n[4] Distribución del voto:")
print(df_consolidado['voto'].value_counts())
print(f"\n[4] Proyectos únicos: {df_consolidado['titulo_base'].nunique():,}")
print(f"[4] Diputados únicos: {df_consolidado['diputado'].nunique():,}")

[1] Registros originales:          578,507
[1] Registros tras eliminar ruido: 544,840
[1] Ruido eliminado:               33,667

[2] Títulos únicos originales: 2,107
[2] Títulos base únicos:        1,802

[3] Registros con voto 'En General': 7,196

[4] Registros originales:          578,507
[4] Tras eliminar ruido:            544,840
[4] Tras consolidar:                460,923
[4] Reducción total:                117,584

[4] Fuente de consolidación:
fuente_consolidacion
moda_articulos    454010
en_general          6913
Name: count, dtype: int64

[4] Distribución del voto:
voto
AFIRMATIVO    268458
AUSENTE       101332
NEGATIVO       80921
ABSTENCION     10212
Name: count, dtype: int64

[4] Proyectos únicos: 1,802
[4] Diputados únicos: 2,061


In [12]:
# Crear fecha_base si no existe
df['fecha_votacion'] = pd.to_datetime(df['fecha_votacion'])
df['fecha_base'] = df['fecha_votacion'].dt.date

# Detección de días consecutivos
def tiene_dias_consecutivos(fechas):
    fechas_ord = sorted(fechas)
    for i in range(len(fechas_ord) - 1):
        if (fechas_ord[i+1] - fechas_ord[i]).days == 1:
            return True
    return False

from datetime import date

rango_sesion = df.groupby('titulo_base')['fecha_base'].apply(
    lambda x: tiene_dias_consecutivos(list(x.unique()))
).reset_index()
rango_sesion.columns = ['titulo_base', 'cruce_medianoche']

casos = rango_sesion[rango_sesion['cruce_medianoche']]
print(f"Títulos base con días consecutivos: {len(casos)}")

for _, row in casos.head(10).iterrows():
    fechas = sorted(df[df['titulo_base'] == row['titulo_base']]['fecha_base'].unique())
    print(f"\n  BASE: {row['titulo_base'][:80]}")
    print(f"  FECHAS: {fechas}")

Títulos base con días consecutivos: 5

  BASE: MOCIÓN SOLICITADA POR EL DIP. BORNORONI, GABRIEL.
  FECHAS: [datetime.date(2025, 12, 17), datetime.date(2025, 12, 18)]

  BASE: Presupuesto General de la Administración Nacional para el Ejercicio Fiscal del a
  FECHAS: [datetime.date(1996, 11, 20), datetime.date(1996, 11, 21)]

  BASE: Presupuesto General de la Administración Nacional para el ejercicio fiscal 2001 
  FECHAS: [datetime.date(2000, 11, 29), datetime.date(2000, 11, 30)]

  BASE: Presupuesto General de la Administración Nacional para el ejercicio fiscal del a
  FECHAS: [datetime.date(1997, 11, 27), datetime.date(1997, 11, 28)]

  BASE: Régimen de Reforma Laboral
  FECHAS: [datetime.date(1998, 9, 2), datetime.date(1998, 9, 3)]
